In [1]:
import ee
import geemap
import pandas as pd
from datetime import datetime, timedelta
import time
from tqdm import tqdm
import logging
import os

# 设置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- 可配置参数 ---
CSV_PATH = r"E:\NWP\CS2_S1_matched\time_match_2023.csv"   # 本地CSV，需包含sceneName列
SCENE_COL = "sceneName"                                    # S1产品名列名
S2_CLOUD_MAX = 20                                          # S2 影像层面云量（%）上限
L8_CLOUD_MAX = 20                                          # L8/9 影像层面云量（%）上限
TIME_WINDOW_HOURS = 24                                      # 时间窗口：±1小时
LIMIT_PER_SENSOR = 1                                       # 每个S1匹配各传感器保留多少条（按云量最小）
DO_DRIVE_EXPORT = False                                    # 是否导出Drive（默认False，先确认匹配结果）
S2_EXPORT_SCALE = 20                                       # Sentinel-2 导出分辨率
L8_EXPORT_SCALE = 30                                       # Landsat 导出分辨率

# 输出文件路径
OUTPUT_DIR = r"E:\NWP\CS2_S1_matched\S1_S2_overlap"                             # 输出目录
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    
# --- 初始化EE账户（会弹浏览器登录一次） ---
geemap.ee_initialize()


In [2]:
class SatelliteImageMatcher:
    """卫星影像匹配类"""
    
    def __init__(self, time_window_hours=2, cloud_threshold=20):
        """
        初始化匹配器
        
        Args:
            time_window_hours: 时间窗口（小时）
            cloud_threshold: 云量阈值（百分比）
        """
        # 初始化Earth Engine
        try:
            ee.Initialize()
            print("✓ Earth Engine 初始化成功")
        except Exception as e:
            print(f"✗ Earth Engine 初始化失败: {e}")
            raise
        
        self.time_window_hours = time_window_hours
        self.cloud_threshold = cloud_threshold
        self.results = []
        
    def load_csv_asset(self, asset_path, scene_name_column='sceneName'):
        """
        加载CSV资产并提取场景名称
        
        Args:
            asset_path: GEE中CSV资产的路径
            scene_name_column: 场景名称列的名称
            
        Returns:
            list: 场景名称列表
        """
        try:
            # 加载CSV作为FeatureCollection
            csv_fc = ee.FeatureCollection(asset_path)
            
            # 获取场景名称列表
            scene_names = csv_fc.aggregate_array(scene_name_column).getInfo()
            
            print(f"✓ 成功加载CSV资产: {len(scene_names)} 个场景名称")
            return scene_names
            
        except Exception as e:
            logger.error(f"加载CSV资产失败: {e}")
            raise
    
    def load_csv_file(self, csv_file_path, scene_name_column='sceneName'):
        """
        从本地CSV文件加载场景名称
        
        Args:
            csv_file_path: 本地CSV文件路径
            scene_name_column: 场景名称列的名称
            
        Returns:
            list: 场景名称列表
        """
        try:
            df = pd.read_csv(csv_file_path)
            scene_names = df[scene_name_column].dropna().tolist()
            
            print(f"✓ 成功加载本地CSV文件: {len(scene_names)} 个场景名称")
            return scene_names
            
        except Exception as e:
            logger.error(f"加载CSV文件失败: {e}")
            raise
    
    def find_sentinel1_image(self, scene_name, roi=None):
        """
        查找Sentinel-1影像
        
        Args:
            scene_name: 场景名称
            roi: 感兴趣区域
            
        Returns:
            dict: Sentinel-1影像信息
        """
        try:
            # 构建Sentinel-1集合
            s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD')
            
            # 尝试不同的筛选方式
            filters = [
                ee.Filter.eq('system:index', scene_name),
                ee.Filter.stringContains('system:id', scene_name),
                ee.Filter.stringContains('PRODUCT_ID', scene_name)
            ]
            
            s1_image = None
            for filter_condition in filters:
                temp_collection = s1_collection.filter(filter_condition).filter(ee.Filter.eq('instrumentMode', 'EW'))
                
                if roi:
                    temp_collection = temp_collection.filterBounds(roi)
                
                if temp_collection.size().getInfo() > 0:
                    s1_image = temp_collection.first()
                    break
            
            if s1_image is None:
                return {'error': f'未找到场景名称为 {scene_name} 的EW模式Sentinel-1影像'}
            
            # 获取影像信息
            image_info = s1_image.getInfo()
            acquisition_time = ee.Date(s1_image.get('system:time_start'))
            
            return {
                'image': s1_image,
                'image_id': image_info['id'],
                'acquisition_time': acquisition_time,
                'acquisition_time_str': acquisition_time.format('YYYY-MM-dd HH:mm:ss').getInfo(),
                'properties': image_info.get('properties', {}),
                'geometry': s1_image.geometry()
            }
            
        except Exception as e:
            return {'error': f'处理场景 {scene_name} 时出错: {str(e)}'}
    
    def find_matching_images(self, s1_info, sensor_type='sentinel2'):
        """
        查找时间窗口内的匹配影像
        
        Args:
            s1_info: Sentinel-1影像信息
            sensor_type: 'sentinel2' 或 'landsat8'
            
        Returns:
            list: 匹配的影像列表
        """
        if 'error' in s1_info:
            return []
        
        try:
            acquisition_time = s1_info['acquisition_time']
            geometry = s1_info['geometry']
            
            # 计算时间窗口
            time_start = acquisition_time.advance(-self.time_window_hours, 'hour')
            time_end = acquisition_time.advance(self.time_window_hours, 'hour')
            
            if sensor_type == 'sentinel2':
                collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                    .filterDate(time_start, time_end) \
                    .filterBounds(geometry) \
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', self.cloud_threshold))
                    
            elif sensor_type == 'landsat8':
                collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
                    .filterDate(time_start, time_end) \
                    .filterBounds(geometry) \
                    .filter(ee.Filter.lt('CLOUD_COVER', self.cloud_threshold))
            else:
                return []
            
            collection_size = collection.size().getInfo()
            
            if collection_size == 0:
                return []
            
            # 限制返回数量以避免内存问题
            max_images = min(collection_size, 10)
            image_list = collection.limit(max_images).getInfo()
            
            matched_images = []
            for img_info in image_list['features']:
                img_time = ee.Date(img_info['properties']['system:time_start'])
                time_diff = img_time.difference(acquisition_time, 'hour').getInfo()
                
                matched_images.append({
                    'image_id': img_info['id'],
                    'acquisition_time': img_time.format('YYYY-MM-dd HH:mm:ss').getInfo(),
                    'time_difference_hours': round(time_diff, 2),
                    'properties': img_info['properties'],
                    'sensor_type': sensor_type
                })
            
            return matched_images
            
        except Exception as e:
            logger.error(f"查找匹配影像时出错: {e}")
            return []
    
    def process_single_scene(self, scene_name, roi=None):
        """
        处理单个场景
        
        Args:
            scene_name: 场景名称
            roi: 感兴趣区域
            
        Returns:
            list: 处理结果列表
        """
        scene_results = []
        
        # 查找Sentinel-1影像
        s1_info = self.find_sentinel1_image(scene_name, roi)
        
        if 'error' in s1_info:
            scene_results.append({
                's1_scene_name': scene_name,
                'error': s1_info['error'],
                'status': 'failed'
            })
            return scene_results
        
        # 查找Sentinel-2匹配
        s2_matches = self.find_matching_images(s1_info, 'sentinel2')
        
        # 查找Landsat-8匹配
        l8_matches = self.find_matching_images(s1_info, 'landsat8')
        
        # 记录Sentinel-2匹配结果
        if s2_matches:
            for s2_match in s2_matches:
                scene_results.append({
                    's1_scene_name': scene_name,
                    's1_image_id': s1_info['image_id'],
                    's1_acquisition_time': s1_info['acquisition_time_str'],
                    'matched_image_id': s2_match['image_id'],
                    'matched_acquisition_time': s2_match['acquisition_time'],
                    'time_difference_hours': s2_match['time_difference_hours'],
                    'sensor_pair': 'Sentinel-1 & Sentinel-2',
                    'matched_sensor': 'Sentinel-2',
                    'status': 'success'
                })
        
        # 记录Landsat-8匹配结果
        if l8_matches:
            for l8_match in l8_matches:
                scene_results.append({
                    's1_scene_name': scene_name,
                    's1_image_id': s1_info['image_id'],
                    's1_acquisition_time': s1_info['acquisition_time_str'],
                    'matched_image_id': l8_match['image_id'],
                    'matched_acquisition_time': l8_match['acquisition_time'],
                    'time_difference_hours': l8_match['time_difference_hours'],
                    'sensor_pair': 'Sentinel-1 & Landsat-8',
                    'matched_sensor': 'Landsat-8',
                    'status': 'success'
                })
        
        # 如果没有找到任何匹配
        if not s2_matches and not l8_matches:
            scene_results.append({
                's1_scene_name': scene_name,
                's1_image_id': s1_info['image_id'],
                's1_acquisition_time': s1_info['acquisition_time_str'],
                'matched_image_id': 'No match found',
                'sensor_pair': 'Sentinel-1 only',
                'status': 'no_match'
            })
        
        return scene_results
    
    def process_batch(self, scene_names, roi=None, batch_size=50, delay=0.5):
        """
        批量处理场景名称
        
        Args:
            scene_names: 场景名称列表
            roi: 感兴趣区域
            batch_size: 批处理大小
            delay: 批次间延迟（秒）
            
        Returns:
            list: 所有处理结果
        """
        all_results = []
        total_scenes = len(scene_names)
        
        print(f"开始批量处理 {total_scenes} 个场景...")
        
        # 分批处理
        for i in tqdm(range(0, total_scenes, batch_size), desc="批次处理进度"):
            batch_scenes = scene_names[i:i+batch_size]
            batch_results = []
            
            # 处理当前批次
            for scene_name in tqdm(batch_scenes, desc=f"批次 {i//batch_size + 1}", leave=False):
                try:
                    scene_results = self.process_single_scene(scene_name, roi)
                    batch_results.extend(scene_results)
                    
                    # 小延迟避免API限制
                    time.sleep(0.1)
                    
                except Exception as e:
                    logger.error(f"处理场景 {scene_name} 时出错: {e}")
                    batch_results.append({
                        's1_scene_name': scene_name,
                        'error': str(e),
                        'status': 'failed'
                    })
            
            all_results.extend(batch_results)
            
            # 批次间延迟
            if i + batch_size < total_scenes:
                time.sleep(delay)
            
            # 定期输出进度
            if (i // batch_size + 1) % 10 == 0:
                print(f"已完成 {len(all_results)} 个结果")
        
        self.results = all_results
        return all_results
    
    def save_results(self, output_path='satellite_matching_results.csv'):
        """
        保存结果到CSV文件
        
        Args:
            output_path: 输出文件路径
        """
        if not self.results:
            print("没有结果可保存")
            return
        
        df = pd.DataFrame(self.results)
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"✓ 结果已保存到: {output_path}")
        
        # 输出统计信息
        self.print_statistics()
    
    def print_statistics(self):
        """输出统计信息"""
        if not self.results:
            return
        
        df = pd.DataFrame(self.results)
        
        print("\n=== 处理统计 ===")
        print(f"总处理数量: {len(df)}")
        
        status_counts = df['status'].value_counts()
        for status, count in status_counts.items():
            print(f"{status}: {count}")
        
        if 'sensor_pair' in df.columns:
            print("\n=== 传感器配对统计 ===")
            pair_counts = df['sensor_pair'].value_counts()
            for pair, count in pair_counts.items():
                print(f"{pair}: {count}")

In [8]:
matcher = SatelliteImageMatcher(time_window_hours=24, cloud_threshold=20)

✓ Earth Engine 初始化成功


In [9]:
scene_names = [
    'S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378',
    'S1A_EW_GRDM_1SDH_20230105T212255_20230105T212334_046654_05978A_50D1',
    # 添加更多场景名称...
]
   # roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
roi = None  # 不限制区域

# 批量处理
results = matcher.process_batch(
    scene_names=scene_names,
    roi=roi,
    batch_size=20,  # 每批处理20个
    delay=1.0       # 批次间延迟1秒
)


开始批量处理 2 个场景...


批次处理进度: 100%|██████████| 1/1 [00:13<00:00, 13.82s/it]


In [10]:
print(results)

[{'s1_scene_name': 'S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378', 's1_image_id': 'COPERNICUS/S1_GRD/S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378', 's1_acquisition_time': '2023-03-29 15:51:10', 'matched_image_id': 'COPERNICUS/S2_SR_HARMONIZED/20230329T210051_20230329T210046_T10XDL', 'matched_acquisition_time': '2023-03-29 21:03:27', 'time_difference_hours': 5.2, 'sensor_pair': 'Sentinel-1 & Sentinel-2', 'matched_sensor': 'Sentinel-2', 'status': 'success'}, {'s1_scene_name': 'S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378', 's1_image_id': 'COPERNICUS/S1_GRD/S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378', 's1_acquisition_time': '2023-03-29 15:51:10', 'matched_image_id': 'COPERNICUS/S2_SR_HARMONIZED/20230329T210051_20230329T210046_T10XDM', 'matched_acquisition_time': '2023-03-29 21:03:17', 'time_difference_hours': 5.2, 'sensor_pair': 'Sentinel-1 & Sentinel-2', 'matched_sensor': 'Sentinel-2', 'status':

In [33]:
print("正在读取CSV文件...")
df = pd.read_csv(CSV_PATH)
if SCENE_COL not in df.columns:
    raise ValueError(f"CSV中未找到列：{SCENE_COL}")

# 筛选EW模式的S1数据
print("筛选EW模式的S1数据...")
ew_scenes = df[df[SCENE_COL].apply(is_ew_mode)][SCENE_COL].dropna().astype(str).drop_duplicates().tolist()

print(f"原始S1数据总数: {len(df[SCENE_COL].dropna())}")
print(f"EW模式S1数据: {len(ew_scenes)}")

if len(ew_scenes) == 0:
    print("未找到EW模式的S1数据，请检查数据")


# 主循环
print("开始检索重叠影像...")

正在读取CSV文件...
筛选EW模式的S1数据...
原始S1数据总数: 360
EW模式S1数据: 207
开始检索重叠影像...


In [34]:
all_records = []
for k, name in enumerate(ew_scenes, 1):
    print(f"[{k}/{len(ew_scenes)}] S1 EW: {name}")
    try:
        recs = find_overlaps_for_s1(name)
        all_records.extend(recs)
    except Exception as e:
        print(f"[ERR] {name}: {e}")
    # 稍作延时，避免过快请求
    time.sleep(0.2)

[1/207] S1 EW: S1A_EW_GRDM_1SDH_20230329T155110_20230329T155214_047861_05C03E_8378
[2/207] S1 EW: S1A_EW_GRDM_1SDH_20230105T212255_20230105T212334_046654_05978A_50D1
[3/207] S1 EW: S1A_EW_GRDM_1SDH_20230630T134853_20230630T134953_049216_05EB02_4620
[4/207] S1 EW: S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83
[5/207] S1 EW: S1A_EW_GRDM_1SDH_20230630T120922_20230630T121027_049215_05EAFB_1952
[6/207] S1 EW: S1A_EW_GRDM_1SDH_20230629T144524_20230629T144629_049202_05EA9F_5AF6
[7/207] S1 EW: S1A_EW_GRDM_1SDH_20230629T130642_20230629T130747_049201_05EA98_7E6F
[8/207] S1 EW: S1A_EW_GRDM_1SDH_20230628T154303_20230628T154357_049188_05EA31_6E24
[9/207] S1 EW: S1A_EW_GRDM_1SDH_20230628T140418_20230628T140522_049187_05EA28_E105
[10/207] S1 EW: S1A_EW_GRDM_1SDH_20230627T114601_20230627T114701_049171_05E9B5_5AF7
[11/207] S1 EW: S1A_EW_GRDM_1SDH_20230626T142248_20230626T142348_049158_05E944_BA88
[12/207] S1 EW: S1A_EW_GRDM_1SDH_20230626T142148_20230626T142248_049158_05E944_ECF8
[

In [30]:
# 结果处理和保存
res_df = pd.DataFrame(all_records)
if len(res_df) == 0:
    print("未找到任何满足 1 小时内&质量阈值 的 S2/L8 匹配。可放宽云量或时间窗口。")


In [12]:
# 按 S1 + 传感器 分组取云量最小TOP1
res_best = (res_df.sort_values(['s1_scene', 'sensor', 'cloud_%'])
            .groupby(['s1_scene', 'sensor'], as_index=False).head(1))

KeyError: 's1_scene'